# Anachronism filter (1930s): thin runner for the `scripts/` pipeline

This notebook is a **thin orchestrator**. All the real logic lives in standalone
Python scripts that are stored in the destination dataset repo under `scripts/`.
The notebook uploads them to Hugging Face, clones the repo, and runs each stage as
its own process. This keeps the heavy code (the 460+ term banned list, the fast
matcher, the shard loop) out of the notebook and versioned alongside the data.

- Source:      [`jbduran/think-dataset-clean`](https://huggingface.co/datasets/jbduran/think-dataset-clean)
- Destination: [`jbduran/think-dataset-clean-1930s`](https://huggingface.co/datasets/jbduran/think-dataset-clean-1930s) (created on first run)
- Method: Michael Hla's keyword-filter approach (drop a whole document that mentions
  anything post-1930), seeded from croqaz/vintage-ft-v1 `banned.txt` and adjusted
  from a 1900 to a 1930 cutoff via an allow-list.

## The pipeline scripts (in `scripts/`)

| Script | Stage | What it does |
|---|---|---|
| `config.py` | — | Shared settings (repos, cutoff, scan window). Imported by all; reads env overrides. |
| `common.py` | — | HF auth, `HfApi`, repo/shard helpers. |
| `wipe.py` | 0 | Guarded clean command: delete generated shards/stats/hits/report. |
| `build_list.py` | 1 | Build + count + upload the banned list to `_banned/`. |
| `footer_lib.py` | — | Footer/boilerplate line detectors; imported by `strip_footers.py`. |
| `strip_footers.py` | 2 | Line-level footer/boilerplate strip; writes the `stripped/` layer. |
| `filter_lib.py` | — | Fast matcher (`compile_matchers`, `scan_text`, `should_drop`); imported by `run_filter.py`. |
| `run_filter.py` | 3 | Resumable shard loop: scan, drop whole docs, write shard + stats + hit log. |
| `report.py` | 4 | Aggregate report + README; ranks which terms fired. |

## How state passes between stages

Each script is its own process. They share settings through `config.py` (with env
overrides set in the bootstrap cell) and share the banned list through HF: Stage 1
uploads `_banned/banned_list.txt`; Stages 3 and 4 download it and rebuild the matcher.
That makes every stage independently runnable and naturally resumable.

## Performance note

The matcher deliberately avoids a 400+ term regex alternation (which is
seconds-per-book on multi-MB OCR text). It uses set membership for single words,
first-token gating for phrases, and a capped head+tail scan window — about 35x
faster with identical results. This runs comfortably on a **plain CPU** runtime;
no GPU, no high-RAM needed.


## 1. Install, authenticate, configure

Installs dependencies, logs into Hugging Face (add a **write** token as a Colab
secret named `HF_TOKEN`), exports config as environment variables for the scripts,
and creates the destination repo.


In [2]:
# === Install dependencies ===================================================
%pip -q install -U datasets huggingface_hub pyarrow tqdm numpy

import os
from pathlib import Path

from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
except Exception:
    userdata = None

# === Authenticate to Hugging Face ==========================================
# Preferred: add a WRITE token as a Colab secret named HF_TOKEN.
HF_TOKEN = None
if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = HfApi().token

# Export so the stage scripts (separate processes) can authenticate.
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=True)

# === Configuration (exported as env vars for the scripts) ==================
DST_REPO = "jbduran/think-dataset-clean-1930s"
SRC_REPO = "jbduran/think-dataset-clean"
os.environ["SRC_REPO"] = SRC_REPO
os.environ["DST_REPO"] = DST_REPO
os.environ["WORK_DIR"] = "/content/think_1930s_work"

# Scan window + policy (see scripts/config.py for meaning). Tweak here if needed.
os.environ["CUTOFF_YEAR"] = "1930"
os.environ["MIN_BANNED_HITS"] = "1"
os.environ["SCAN_CHARS"] = "300000"
os.environ["SCAN_TAIL_CHARS"] = "50000"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=DST_REPO, repo_type="dataset", exist_ok=True, private=False)
print(f"Authenticated. Destination repo ready: {DST_REPO}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 84.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated. Destination repo ready: jbduran/think-dataset-clean-1930s


## 2. Push scripts to HF and clone

Bring the local `scripts/` folder into this Colab session first (drag-and-drop the
folder into the file browser on the left, or mount Google Drive). This cell uploads
it to `scripts/` in the destination repo, then clones the repo so every stage runs
from a clean checkout. Re-running re-syncs the scripts and pulls the latest.


In [3]:
# === Push the pipeline scripts to Hugging Face, then clone the repo =========
# The pipeline lives in a local `scripts/` folder. Bring it into this Colab
# session first (drag-and-drop the folder into the file browser, or mount Drive).
# This cell uploads it to `scripts/` in the destination repo, then clones the
# repo so every stage runs from a clean checkout on HF.

LOCAL_SCRIPTS = Path("scripts")   # adjust if you placed it elsewhere in Colab

assert LOCAL_SCRIPTS.is_dir(), (
    f"Could not find '{LOCAL_SCRIPTS}/' in this Colab session. Upload the scripts "
    "folder (config.py, common.py, build_list.py, filter_lib.py, run_filter.py, "
    "report.py, wipe.py) into the working directory and re-run this cell."
)

api.upload_folder(
    repo_id=DST_REPO,
    repo_type="dataset",
    folder_path=str(LOCAL_SCRIPTS),
    path_in_repo="scripts",
    commit_message="upload/update 1930s pipeline scripts",
)
print("Uploaded scripts/ to HF.")

# Clone the repo (or pull if already cloned) and run everything from there.
CLONE_DIR = Path("/content/think-dataset-clean-1930s")
if CLONE_DIR.exists():
    !cd "{CLONE_DIR}" && git pull --quiet
else:
    !git clone --quiet "https://huggingface.co/datasets/{DST_REPO}" "{CLONE_DIR}"

RUN_DIR = CLONE_DIR / "scripts"
print(f"Running stages from: {RUN_DIR}")
!ls -la "{RUN_DIR}"

Uploaded scripts/ to HF.
Running stages from: /content/think-dataset-clean-1930s/scripts
total 100
drwxr-xr-x 2 root root  4096 Jul  8 16:12 .
drwxr-xr-x 8 root root  4096 Jul  8 16:12 ..
-rw-r--r-- 1 root root 27114 Jul  8 16:12 build_list.py
-rw-r--r-- 1 root root  5483 Jul  8 16:12 common.py
-rw-r--r-- 1 root root  4254 Jul  8 16:12 config.py
-rw-r--r-- 1 root root  8222 Jul  8 16:12 filter_lib.py
-rw-r--r-- 1 root root  5296 Jul  8 16:12 footer_lib.py
-rw-r--r-- 1 root root  7730 Jul  8 16:12 report.py
-rw-r--r-- 1 root root  7470 Jul  8 16:12 run_filter.py
-rw-r--r-- 1 root root  6926 Jul  8 16:12 strip_footers.py
-rw-r--r-- 1 root root  2864 Jul  8 16:12 wipe.py


## 3. Stage 0 — clean command (optional, guarded)

Only deletes anything if you edit the cell to set `CONFIRM_WIPE=1`. Use it to
rebuild the dataset from scratch. Leave as-is for a normal resumable run.


In [ ]:
# === Stage 0: clean command (guarded) =======================================
#   CONFIRM_WIPE=1       Wipes all of the filtered shards
#   WIPE_BANNED_LIST=1   Wipes the banned list of words
#   WIPE_STRIPPED=1      Wipes all of the stripped shards
!cd "{CLONE_DIR}" && CONFIRM_WIPE=1 WIPE_BANNED_LIST=0 WIPE_STRIPPED=1 python scripts/wipe.py

## 4. Stage 1 — build the banned list

Builds the list (croqaz seed + 1930 allow-list + post-1930 extras), prints the
**total term count**, and uploads it to `_banned/`. Subsequent runs load the cached
list unless you set `FORCE_REBUILD_LIST=1`.


In [ ]:
# === Stage 1: build (or load) the banned list ===============================
# Builds the 1930s banned list from croqaz + allow-list + extras, prints the
# total term count, and uploads it under _banned/ in the destination repo.
# Re-running loads the cached list; set FORCE_REBUILD_LIST=1 to rebuild.
!cd "{CLONE_DIR}" && FORCE_REBUILD_LIST=0 python scripts/build_list.py

## 5. Stage 2 — strip footers
Footer/boilerplate removal runs **before** the anachronism filter. This line-level
pass removes reprint/OCR footer lines — URLs, "printed in the United States of
America", "all rights reserved", photocopy / print-on-demand colophons, ISBN lines,
bare page numbers, library stamps — from each document, writing the stripped corpus
to `stripped/` in the destination repo. Whole books are kept; only footer lines go.

Once the strip samples look right, strip the rest of the corpus. Already-stripped
shards are skipped, so re-run after any disconnect until the whole corpus is done.
Only then move on to the anachronism filter below, which reads the `stripped/` layer.


In [ ]:
# === Stage 2 (full run): strip footers from all remaining shards ============
# After you've inspected the dry-run strip samples and are happy, run this to
# strip the rest of the corpus. Already-stripped shards are skipped, so it's safe
# to re-run after any Colab disconnect until the whole corpus is stripped. Only
# then move on to the anachronism filter (Stage 3), which reads `stripped/`.

# test run (1 shard)
!cd "{CLONE_DIR}" && BATCH_SIZE=1 DRY_RUN_LIMIT=1 python scripts/strip_footers.py

# full run (0 = all shards)
#!cd "{CLONE_DIR}" && BATCH_SIZE=25 DRY_RUN_LIMIT=0 python scripts/strip_footers.py

## 6. Stage 3 — anachronism filter

Once the dry-run hit log looks right, run this to process everything. Completed
shards are skipped, so it resumes automatically after any Colab disconnect — just
re-run this cell until it reports nothing remaining. Each shard prints kept/removed
counts and a running ETA.


In [1]:
# === Stage 3 (full run): process all remaining shards =======================
# test run (1 shard)
!cd "{CLONE_DIR}" && SRC_REPO="{DST_REPO}" SRC_PREFIX=stripped DRY_RUN_LIMIT=1 BATCH_SIZE=1 python scripts/run_filter.py

# full run (0 = all shards)
#!cd "{CLONE_DIR}" && SRC_REPO="{DST_REPO}" SRC_PREFIX=stripped DRY_RUN_LIMIT=0 BATCH_SIZE=25 python scripts/run_filter.py

/bin/bash: line 1: cd: {CLONE_DIR}: No such file or directory


## 7. Stage 4 — aggregate report

Summarizes all completed shards, ranks the top firing terms and the top footer
patterns (your audit surfaces), and writes `cleaning_report_1930s.json` + `README.md`
to the destination repo. Safe to run at any point during the run.


In [ ]:
# === Stage 4: aggregate report + README =====================================
# Sums all per-shard stats, ranks which banned terms actually fired, and writes
# cleaning_report_1930s.json + README.md to the destination repo. Safe to run at
# any point -- it reports whatever shards have completed so far.
!cd "{CLONE_DIR}" && python scripts/report.py

## 8. Visualizations

Run after the pipeline (or partway through) to inspect results. Each graph reads
the per-shard stats / file listing from the destination repo on HF. Requires the
bootstrap cell (for HF_TOKEN, SRC_REPO, DST_REPO) to have run.

### Graph 1 - shard sizes: original vs stripped vs anachronism-filtered

Overlaid line per layer showing each shard's parquet size (MB) vs shard index.
Shows how much each stage shrinks the corpus.

In [ ]:
# === Graph 1: shard sizes -- original vs stripped vs filtered ===============
# Reads the HF file listing (no downloads) and plots each shard's parquet byte
# size for the three layers as overlaid lines vs shard index. Shows how much each
# stage shrinks the corpus. (473 shards -> lines are far more legible than bars.)
import re
import matplotlib.pyplot as plt
from huggingface_hub import HfApi

_api = HfApi(token=HF_TOKEN)
_info = _api.list_repo_files  # noqa (kept for clarity)

# repo_info with files=... gives sizes; fall back gracefully if unavailable.
_ri = _api.repo_info(repo_id=DST_REPO, repo_type="dataset", files_metadata=True)
_src_ri = _api.repo_info(repo_id=SRC_REPO, repo_type="dataset", files_metadata=True)

def _sizes(repo_info, prefix):
    """Return {shard_index: size_MB} for shard_XXXXX.parquet under `prefix`."""
    out = {}
    pref = (prefix.rstrip("/") + "/") if prefix else ""
    for f in repo_info.siblings:
        name = f.rfilename
        m = re.match(rf"^{re.escape(pref)}shard_(\d+)\.parquet$", name)
        if m and "/" not in name[len(pref):]:  # exactly at this level
            size = getattr(f, "size", None)
            if size:
                out[int(m.group(1))] = size / 1e6
    return out

original = _sizes(_src_ri, "")           # jbduran/think-dataset-clean (root)
stripped = _sizes(_ri, "stripped")       # stripped/ layer in dst
filtered = _sizes(_ri, "")               # final filtered shards at dst root

idx = sorted(set(original) | set(stripped) | set(filtered))
xo = sorted(original); yo = [original[i] for i in xo]
xs = sorted(stripped); ys = [stripped[i] for i in xs]
xf = sorted(filtered); yf = [filtered[i] for i in xf]

fig, ax = plt.subplots(figsize=(13, 5))
if xo: ax.plot(xo, yo, lw=0.9, alpha=0.9, label=f"original ({len(xo)} shards)")
if xs: ax.plot(xs, ys, lw=0.9, alpha=0.9, label=f"stripped ({len(xs)} shards)")
if xf: ax.plot(xf, yf, lw=0.9, alpha=0.9, label=f"filtered ({len(xf)} shards)")
ax.set_xlabel("shard index")
ax.set_ylabel("parquet size (MB)")
ax.set_title("Shard size by pipeline stage: original vs stripped vs anachronism-filtered")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# quick corpus-level summary
def _tot(d): return sum(d.values())
print(f"Total size  original: {_tot(original):8.1f} MB")
print(f"            stripped: {_tot(stripped):8.1f} MB  "
      f"({100*(1-_tot(stripped)/max(_tot(original),1e-9)):.2f}% smaller)" if stripped else "            stripped: (not run yet)")
print(f"            filtered: {_tot(filtered):8.1f} MB  "
      f"({100*(1-_tot(filtered)/max(_tot(original),1e-9)):.2f}% smaller)" if filtered else "            filtered: (not run yet)")

### Graph 2 - footer-strip methods compared

Total footer lines removed per pattern across all shards, sorted. Highest bar is
highlighted red, lowest green.

In [ ]:
# === Graph 2: footer-strip methods -- total lines removed per pattern ========
# Aggregates removed_by_pattern across every strip_stats/*.json and plots one bar
# per footer pattern, sorted. The highest and lowest patterns are highlighted.
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

_api = HfApi(token=HF_TOKEN)
_files = _api.list_repo_files(repo_id=DST_REPO, repo_type="dataset")
_strip_stats = sorted(f for f in _files if re.match(r"strip_stats/shard_\d+\.json$", f))
print(f"Aggregating {len(_strip_stats)} strip_stats files...")

_totals = Counter()
for f in _strip_stats:
    local = hf_hub_download(DST_REPO, f, repo_type="dataset", token=HF_TOKEN)
    with open(local, encoding="utf-8") as fh:
        st = json.load(fh)
    _totals.update(st.get("removed_by_pattern", {}))

if not _totals:
    print("No strip stats found yet -- run Stage 2 (strip footers) first.")
else:
    items = _totals.most_common()                 # sorted high -> low
    labels = [k for k, _ in items]
    values = [v for _, v in items]

    # highlight highest (first) and lowest (last)
    colors = ["#4C78A8"] * len(values)
    colors[0] = "#E45756"                          # highest -> red
    colors[-1] = "#59A14F"                         # lowest  -> green

    fig, ax = plt.subplots(figsize=(11, max(4, 0.42 * len(labels))))
    bars = ax.barh(labels, values, color=colors)
    ax.invert_yaxis()                              # highest at top
    ax.set_xlabel("total footer lines removed (all shards)")
    ax.set_title("Footer-strip methods compared (highest = red, lowest = green)")
    for b, v in zip(bars, values):
        ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f" {v:,}",
                va="center", fontsize=9)
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"HIGHEST: {labels[0]} = {values[0]:,} lines")
    print(f"LOWEST:  {labels[-1]} = {values[-1]:,} lines")
    print(f"Total footer lines removed: {sum(values):,}")

### Graph 3 - anachronism filter: top firing terms

The 9 banned terms responsible for the most document drops, plus an "Other" bar
summing every remaining term.

In [ ]:
# === Graph 3: anachronism filter -- which banned terms fired ================
# Aggregates removed_by_term across every stats/*.json (the terms responsible for
# dropping documents). Plots the TOP 9 terms as their own bars, with all remaining
# terms summed into a single "Other" bar.
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

_api = HfApi(token=HF_TOKEN)
_files = _api.list_repo_files(repo_id=DST_REPO, repo_type="dataset")
_stats = sorted(f for f in _files if re.match(r"stats/shard_\d+\.json$", f))
print(f"Aggregating {len(_stats)} filter stats files...")

_totals = Counter()
for f in _stats:
    local = hf_hub_download(DST_REPO, f, repo_type="dataset", token=HF_TOKEN)
    with open(local, encoding="utf-8") as fh:
        st = json.load(fh)
    _totals.update(st.get("removed_by_term", {}))

if not _totals:
    print("No filter stats found yet -- run Stage 3 (anachronism filter) first.")
else:
    TOP_N = 9
    top = _totals.most_common(TOP_N)
    other = sum(v for _, v in _totals.most_common()[TOP_N:])
    n_other_terms = max(0, len(_totals) - TOP_N)

    labels = [k for k, _ in top]
    values = [v for _, v in top]
    if other > 0:
        labels.append(f"Other ({n_other_terms} terms)")
        values.append(other)

    colors = ["#4C78A8"] * len(values)
    if other > 0:
        colors[-1] = "#BAB0AC"                    # "Other" -> grey

    fig, ax = plt.subplots(figsize=(11, 5.5))
    bars = ax.bar(labels, values, color=colors)
    ax.set_ylabel("documents dropped")
    ax.set_title(f"Anachronism filter: top {TOP_N} firing terms (+ Other)")
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}",
                ha="center", va="bottom", fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Top {TOP_N} terms account for {sum(values) - other:,} drops; "
          f"'Other' = {other:,} across {n_other_terms} terms.")
    print(f"Total documents dropped by the anachronism filter: {sum(_totals.values()):,}")